# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UsmanRizwan20/week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'fact_month':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Rebuild the same feature frame as w03: days 1-15 = features, days 16-end = label window
data = con.sql(f"""
    WITH bounds AS (SELECT DATE '2026-03-15' AS split_d),
    agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= b.split_d THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date >  b.split_d THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(CASE WHEN report_date <= b.split_d THEN gsc_clicks ELSE 0 END)      AS clicks_first_half,
            AVG(CASE WHEN report_date <= b.split_d AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
            COUNT(DISTINCT CASE WHEN report_date <= b.split_d AND gsc_impressions > 0 THEN report_date END) AS active_days_first_half
        FROM {TABLES['fact_month']} f, bounds b
        GROUP BY 1,2
        HAVING imp_first_half >= 10
    )
    SELECT *, CASE WHEN clicks_first_half > 0 THEN clicks_first_half::FLOAT / imp_first_half ELSE 0 END AS ctr_first_half,
           CASE WHEN imp_second_half < 0.8 * imp_first_half THEN 1 ELSE 0 END AS is_declining_second_half
    FROM agg
""").df()
print(len(data), "content items ready for baseline scoring")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120513 content items ready for baseline scoring


## 1. My rule and its reason codes

Two signal checks first (features only — no imp_second_half / is_declining_second_half used here):

Signal A — position vs CTR (behind FlyRank's CTR-fix logic): claim = "better position associates with higher CTR." Bucket by position tier, weighted CTR (total clicks / total impressions, not mean-of-rates), n printed.
Signal B — volume vs weak CTR (behind quick-win logic): claim = "high-impression pages disproportionately carry the low-CTR-for-position cases worth a quick win." Bucket by volume tier, share flagged, n printed.

The rule, in plain words: "A page is worth a CTR review if it already gets meaningful search volume, ranks well enough that clicks should follow (position ≤ 20), but its CTR sits below what similarly-positioned pages get." Score = visible * good_enough_position * low_ctr_for_tier * imp_first_half — readable, no fitted weights.

Reason code: low_ctr_visible_page. Action label: review_ctr_page

In [3]:
import numpy as np
import pandas as pd
# --- Signal A: position tier vs weighted CTR (flag-linked: CTR-fix logic) ---
d = data.copy()
d['position_tier'] = np.select(
    [d['avg_position_first_half'] <= 3, d['avg_position_first_half'] <= 10,
     d['avg_position_first_half'] <= 20, d['avg_position_first_half'] <= 50],
    ['top_3', 'page_1', 'striking', 'page_3_5'], default='deep_or_no_data')

sigA = (d.groupby('position_tier')
          .apply(lambda g: pd.Series({
              'n': len(g),
              'weighted_ctr': g['clicks_first_half'].sum() / g['imp_first_half'].sum()
          })))
print(sigA)
print("Verdict: CONFIRMED if weighted_ctr clearly falls as position_tier worsens, n>=50 per bucket else 'insufficient data'")

# --- Signal B: volume tier vs low-CTR share (flag-linked: quick-win logic) ---
tier_ctr = d.groupby('position_tier')['clicks_first_half'].transform('sum') / d.groupby('position_tier')['imp_first_half'].transform('sum')
d['low_ctr_for_tier'] = d['ctr_first_half'] < tier_ctr

d['volume_tier'] = pd.qcut(d['imp_first_half'], q=[0, .5, .8, 1.0], labels=['low', 'mid', 'high'])
sigB = d.groupby('volume_tier').agg(n=('content_hash_id', 'size'), low_ctr_share=('low_ctr_for_tier', 'mean'))
print(sigB)
print("Verdict: CONFIRMED if low_ctr_share rises with volume_tier, n>=50 per bucket else 'insufficient data'")

                       n  weighted_ctr
position_tier                         
deep_or_no_data   7578.0      0.000416
page_1           53786.0      0.003282
page_3_5         24763.0      0.001390
striking         23804.0      0.003368
top_3            10582.0      0.004506
Verdict: CONFIRMED if weighted_ctr clearly falls as position_tier worsens, n>=50 per bucket else 'insufficient data'
                 n  low_ctr_share
volume_tier                      
low          60280       0.862177
mid          36144       0.661161
high         24089       0.635477
Verdict: CONFIRMED if low_ctr_share rises with volume_tier, n>=50 per bucket else 'insufficient data'


/tmp/ipykernel_375/1725798981.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/tmp/ipykernel_375/1725798981.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sigB = d.groupby('volume_tier').agg(n=('content_hash_id', 'size'), low_ctr_share=('low_ctr_for_tier', 'mean'))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

d['visible'] = (d['imp_first_half'] >= 100).astype(int)
d['good_position'] = ((d['avg_position_first_half'] > 0) & (d['avg_position_first_half'] <= 20)).astype(int)
d['low_ctr'] = d['low_ctr_for_tier'].astype(int)

d['score'] = d['visible'] * d['good_position'] * d['low_ctr'] * d['imp_first_half']
d['reason_code'] = np.where(d['score'] > 0, 'low_ctr_visible_page', 'no_action')
d['action'] = np.where(d['score'] > 0, 'review_ctr_page', 'monitor')

queue = d.sort_values('score', ascending=False)[
    ['content_hash_id', 'client_hash_id', 'imp_first_half', 'avg_position_first_half',
     'ctr_first_half', 'score', 'reason_code', 'action']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

wrote 120513 rows to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,imp_first_half,avg_position_first_half,ctr_first_half,score,reason_code,action
59435,content_e8a52cf3d5988c07,client_23a62021009f63c4,143173.0,16.018687,0.002466,143173.0,low_ctr_visible_page,review_ctr_page
100296,content_b99ea6861864dea5,client_62f4a7e64f5e0096,91474.0,4.103429,0.002022,91474.0,low_ctr_visible_page,review_ctr_page
39850,content_7c6373141eae744a,client_62f4a7e64f5e0096,86860.0,5.785512,0.000587,86860.0,low_ctr_visible_page,review_ctr_page
87794,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83772.0,8.607910,0.000000,83772.0,low_ctr_visible_page,review_ctr_page
39879,content_acbcc847f8996314,client_62f4a7e64f5e0096,83715.0,3.453361,0.001589,83715.0,low_ctr_visible_page,review_ctr_page
118251,content_471d9cabce329a66,client_73cda7b4e4f265ea,79546.0,4.560527,0.002539,79546.0,low_ctr_visible_page,review_ctr_page
574,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,78162.0,3.647660,0.002597,78162.0,low_ctr_visible_page,review_ctr_page
8101,content_34a70fea29d15f24,client_62f4a7e64f5e0096,73639.0,2.786744,0.000244,73639.0,low_ctr_visible_page,review_ctr_page
59538,content_5e1c049f62e33b11,client_23a62021009f63c4,72940.0,18.233335,0.001481,72940.0,low_ctr_visible_page,review_ctr_page
25370,content_82e35c4845e6c391,client_20259bd6705d81d4,70169.0,18.269589,0.000413,70169.0,low_ctr_visible_page,review_ctr_page


## 3. Top-20 review

For each of the top 10: the action is always review_ctr_page because they all triggered low_ctr_visible_page. What varies is confidence: rows with imp_first_half in the thousands and a CTR far below tier average are high-confidence; rows just barely under the tier CTR with modest volume are low-confidence. What would make each wrong: a page whose low CTR is actually a mismatched-intent query mix rather than a fixable title/meta issue — check fact_content_query_90d before acting.

In [5]:
top10 = queue.head(10).copy()
top10['confidence'] = np.where(top10['imp_first_half'] >= 1000, 'high', 'medium')
for i, row in top10.iterrows():
    print(f"- {row['action']} | reason={row['reason_code']} | imp={row['imp_first_half']:.0f} "
          f"pos={row['avg_position_first_half']:.1f} ctr={row['ctr_first_half']:.3f} "
          f"confidence={row['confidence']} | would be wrong if: intent mismatch, not a CTR-fixable page")
top10

- review_ctr_page | reason=low_ctr_visible_page | imp=143173 pos=16.0 ctr=0.002 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=91474 pos=4.1 ctr=0.002 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=86860 pos=5.8 ctr=0.001 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=83772 pos=8.6 ctr=0.000 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=83715 pos=3.5 ctr=0.002 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=79546 pos=4.6 ctr=0.003 confidence=high | would be wrong if: intent mismatch, not a CTR-fixable page
- review_ctr_page | reason=low_ctr_visible_page | imp=78162 pos=3.6 

,content_hash_id,client_hash_id,imp_first_half,avg_position_first_half,ctr_first_half,score,reason_code,action,confidence
59435,content_e8a52cf3d5988c07,client_23a62021009f63c4,143173.0,16.018687,0.002466,143173.0,low_ctr_visible_page,review_ctr_page,high
100296,content_b99ea6861864dea5,client_62f4a7e64f5e0096,91474.0,4.103429,0.002022,91474.0,low_ctr_visible_page,review_ctr_page,high
39850,content_7c6373141eae744a,client_62f4a7e64f5e0096,86860.0,5.785512,0.000587,86860.0,low_ctr_visible_page,review_ctr_page,high
87794,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83772.0,8.607910,0.000000,83772.0,low_ctr_visible_page,review_ctr_page,high
39879,content_acbcc847f8996314,client_62f4a7e64f5e0096,83715.0,3.453361,0.001589,83715.0,low_ctr_visible_page,review_ctr_page,high
118251,content_471d9cabce329a66,client_73cda7b4e4f265ea,79546.0,4.560527,0.002539,79546.0,low_ctr_visible_page,review_ctr_page,high
574,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,78162.0,3.647660,0.002597,78162.0,low_ctr_visible_page,review_ctr_page,high
8101,content_34a70fea29d15f24,client_62f4a7e64f5e0096,73639.0,2.786744,0.000244,73639.0,low_ctr_visible_page,review_ctr_page,high
59538,content_5e1c049f62e33b11,client_23a62021009f63c4,72940.0,18.233335,0.001481,72940.0,low_ctr_visible_page,review_ctr_page,high
25370,content_82e35c4845e6c391,client_20259bd6705d81d4,70169.0,18.269589,0.000413,70169.0,low_ctr_visible_page,review_ctr_page,high


## 4. Weak picks + leakage check

Weakest pick in the top 10: the one closest to the imp_first_half >= 100 cutoff — low absolute volume means the CTR gap could be one or two clicks of noise, not a real pattern. No leakage: the rule only reads imp_first_half, avg_position_first_half, ctr_first_half, clicks_first_half — all computed from the first-half (feature) window. imp_second_half and is_declining_second_half are never referenced in the scoring cell above, confirmed by inspection.

In [6]:
weakest = top10.iloc[-1]
print("Weakest top-10 pick:")
print(weakest[['content_hash_id', 'imp_first_half', 'avg_position_first_half', 'ctr_first_half']])

leak_check = [c for c in ['imp_second_half', 'is_declining_second_half'] if c in queue.columns]
print("Label/future columns present in queue output (should be empty):", leak_check)

Weakest top-10 pick:
content_hash_id            content_82e35c4845e6c391
imp_first_half                              70169.0
avg_position_first_half                   18.269589
ctr_first_half                             0.000413
Name: 25370, dtype: object
Label/future columns present in queue output (should be empty): []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.